[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# Coordinate Reference Systems &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from pyproj import CRS, Transformer
import matplotlib.pyplot as plt

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


weather_stations = pd.DataFrame({
    "name": ["Bergen", "Oslo", "Svalbard", "Tromso", "Kirkenes"],
    "latitude": [60.39, 59.91, 78.22, 69.65, 69.73],
    "longitude": [5.32, 10.75, 15.65, 18.96, 30.05],
})


def mainland_norway():
    """Norway's outline south of 74 degrees north, without Svalbard, in UTM zone 33N."""
    parts = countries[countries["ADMIN"] == "Norway"].to_crs(25833).explode()
    return parts[parts.geometry.bounds["maxy"] < 8_200_000].union_all()


def station_network(n=300, seed=2026):
    """A made-up register of weather stations on mainland Norway, one row each."""
    rng = np.random.default_rng(seed)
    land = mainland_norway()
    x_min, y_min, x_max, y_max = land.bounds
    east, north = [], []
    while len(east) < n:
        x, y = rng.uniform(x_min, x_max), rng.uniform(y_min, y_max)
        if land.contains(shapely.Point(x, y)):
            east.append(x)
            north.append(y)
    lonlat = gpd.GeoSeries(gpd.points_from_xy(east, north), crs=25833).to_crs(4326)
    return pd.DataFrame({
        "station": [f"NS{i:03d}" for i in range(1, n + 1)],
        "latitude": lonlat.y.round(4).to_numpy(), "longitude": lonlat.x.round(4).to_numpy(),
        "elevation_m": np.clip(rng.gamma(1.6, 220, n), 2, 1900).round(0).astype(int),
        "owner": rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2]),
        "installed": rng.integers(1950, 2025, n),
    })


DISTRICT_CENTERS = {"South": (8.0, 58.6), "East": (10.8, 60.5), "West": (6.0, 60.8), "Mid": (11.0, 63.6),
                    "North": (14.5, 67.0), "Far North": (19.0, 69.4), "Northeast": (25.5, 70.0),
                    "Svalbard": (16.0, 78.5)}


def forecast_districts():
    """Eight made-up forecast districts that cover Norway, in UTM zone 33N."""
    norway = countries[countries["ADMIN"] == "Norway"].to_crs(25833).geometry.iloc[0]
    centers = gpd.GeoSeries(gpd.points_from_xy(*zip(*DISTRICT_CENTERS.values())), crs=4326).to_crs(25833)
    cells = shapely.voronoi_polygons(shapely.MultiPoint(list(centers)),
                                     extend_to=shapely.box(*norway.buffer(200_000).bounds)).geoms
    names = [next(name for name, center in zip(DISTRICT_CENTERS, centers) if cell.contains(center))
             for cell in cells]
    districts = gpd.GeoDataFrame({"district": names}, geometry=[cell.intersection(norway) for cell in cells],
                                 crs=25833)
    return districts.set_index("district").loc[list(DISTRICT_CENTERS)].reset_index()


network = station_network()
stations = gpd.GeoDataFrame(
    network, geometry=gpd.points_from_xy(network["longitude"], network["latitude"]), crs=4326
)


named = gpd.GeoDataFrame(
    weather_stations,
    geometry=gpd.points_from_xy(weather_stations["longitude"], weather_stations["latitude"]),
    crs=4326,
)


districts = forecast_districts()
print("geopandas", gpd.__version__, "| stations:", stations.crs, "| districts:", districts.crs,
      "| countries:", countries.crs)

geopandas 1.2.0 | stations: EPSG:4326 | districts: EPSG:25833 | countries: EPSG:4326


**1.** Make the CRS with EPSG code 3035, a projection used across Europe, and print its name, the unit of its first axis, whether it is projected, and the order of its axes.

In [2]:
europe = CRS(3035)
print(europe.name, "|", europe.axis_info[0].unit_name, "| projected:", europe.is_projected)
print([axis.abbrev for axis in europe.axis_info])

ETRS89-extended / LAEA Europe | metre | projected: True
['Y', 'X']


EPSG:3035 is in meters, and its definition lists the northing, `Y`, first. GeoPandas still stores x first, so a GeoDataFrame converted to it holds eastings as x.

**2.** Convert `named` to UTM zone 33N, and print each station's name with its easting and northing in kilometers, rounded to a whole number.

In [3]:
named_m = named.to_crs(25833)
for name, point in zip(named_m["name"], named_m.geometry):
    print(f"{name:9} {point.x / 1000:6.0f} km east  {point.y / 1000:6.0f} km north")

Bergen       -32 km east    6734 km north
Oslo         262 km east    6649 km north
Svalbard     515 km east    8683 km north
Tromso       654 km east    7732 km north
Kirkenes    1077 km east    7807 km north


Bergen comes out with a negative easting. UTM zone 33N adds 500 km to every easting so that the zone itself stays positive, and Bergen lies 532 km west of the zone's center at 15 degrees east.

**3.** Measure Norway's area in EPSG:3035, which keeps areas true, and print it in square kilometers beside the 386,897 km² that UTM zone 33N gives.

In [4]:
norway = countries[countries["ADMIN"] == "Norway"]
print("EPSG:3035:", round(norway.to_crs(3035).area.iloc[0] / 1e6), "km²")
print("UTM 33N:  ", round(norway.to_crs(25833).area.iloc[0] / 1e6), "km²")

EPSG:3035: 386356 km²
UTM 33N:   386897 km²


The two differ by about 540 km², less than 0.2%. UTM zone 33N stretches areas a little away from its center, and the **Measuring Distance and Area** notebook looks at that stretch.

**4.** Count the made-up stations that lie west of 12 degrees east, the other edge of UTM zone 33N's area of use.

In [5]:
print(int((stations.geometry.x < 12).sum()), "of", len(stations))

175 of 300


`stations` is in WGS 84, so `geometry.x` is the longitude in degrees.

**5.** A file gives Bergen's position as 297,230 m east and 6,700,510 m north in UTM zone 32N, EPSG:25832. Make a GeoSeries from those numbers, give it that CRS, and print its longitude and latitude, rounded to two places.

In [6]:
bergen = gpd.GeoSeries(gpd.points_from_xy([297_230], [6_700_510]), crs=25832)
lonlat = bergen.to_crs(4326).iloc[0]
print(round(lonlat.x, 2), round(lonlat.y, 2))

5.32 60.39


`crs=25832` labels the numbers with the CRS they are in, and `to_crs(4326)` converts them, which gives Bergen's longitude and latitude, 5.32 and 60.39.

---

&#8592; **Back to:** [Coordinate Reference Systems](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/04-coordinate-reference-systems.ipynb)  &nbsp;&middot;&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)
